# Preparación del modelo de clustering para tarjetas gráficas

En este notebook se prepara un modelo de clustering para agrupar tarjetas gráficas con características parecidas.

Partimos de los datos limpios guardados en PostgreSQL y utilizamos variables del producto y de sus especificaciones técnicas.

El objetivo es crear grupos de tarjetas gráficas que ayuden a comparar modelos similares dentro del catálogo.

In [24]:
import pandas as pd
import psycopg

password = input("Contraseña PostgreSQL: ")

conn = psycopg.connect(
    dbname="pccomponentes_ml",
    user="postgres",
    password=password,
    host="localhost",
    port=5432,
)

print("Conexión creada correctamente")

Conexión creada correctamente


## 1. Carga de datos desde PostgreSQL

Cargamos los productos de tarjetas gráficas junto con sus especificaciones técnicas.

La consulta une la tabla general de productos con la tabla de especificaciones GPU para tener en un único DataFrame las variables que se utilizarán en el análisis.

In [25]:
consulta_gpu = """
SELECT
    p.producto_id,
    p.nombre,
    p.marca,
    p.precio,
    p.valoracion_media,
    p.total_opiniones,
    p.porcentaje_recomendacion,
    e.gpu,
    e.memoria_vram,
    e.tipo_memoria,
    e.bus_memoria,
    e.velocidad_memoria,
    e.reloj_base,
    e.reloj_boost,
    e.resolucion_maxima
FROM productos p
JOIN especificaciones_gpu e
    ON p.producto_id = e.producto_id
WHERE p.categoria = 'tarjeta_grafica';
"""

datos_gpu = pd.read_sql(consulta_gpu, conn)

print(datos_gpu.shape)
datos_gpu.head()

(500, 15)


C:\Users\raulr\AppData\Local\Temp\ipykernel_34632\1152358518.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datos_gpu = pd.read_sql(consulta_gpu, conn)


,producto_id,nombre,marca,precio,valoracion_media,total_opiniones,porcentaje_recomendacion,gpu,memoria_vram,tipo_memoria,bus_memoria,velocidad_memoria,reloj_base,reloj_boost,resolucion_maxima
0,gpu_0001,Acer Nitro Radeon RX 9060 XT OC 8GB GDDR6 Edic...,Acer,549.00,0.0,0,0.0,AMD Radeon RX 9060 XT,8 GB GDDR6,GB GDDR6,128 bits,320 GB/s,1900 MHz,3300 MHz,7680 x 4320 píxeles
1,gpu_0002,Acer Nitro Radeon RX 9060 XT OC 8GB GDDR6 HDMI...,Acer,543.00,0.0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gpu_0003,Acer Nitro Radeon RX 9070 OC 16GB GDDR6 PCIe 5...,Acer,715.92,0.0,0,0.0,AMD Radeon RX 9070 OC,16 GB GDDR6,GB GDDR6,NaN,NaN,NaN,NaN,7680 x 4320 píxeles
3,gpu_0004,Acer Predator BiFrost Radeon RX 9070 OC 16GB G...,Acer,802.59,0.0,0,0.0,AMD Radeon RX 9070,16 GB GDDR6,GB GDDR6,256 bits,640 GB/s,1440 MHz,2700 MHz,7680 x 4320 píxeles
4,gpu_0005,Acer Predator BiFrost Radeon RX 9070 OC 16GB G...,Acer,738.53,0.0,0,0.0,AMD Radeon RX 9070 OC,16 GB GDDR6,GB GDDR6,256 bit,NaN,NaN,NaN,7680 x 4320 píxeles (8K)


## 2. Selección de variables para el modelo

Seleccionamos las variables que tienen suficiente información para crear los grupos.

Se descartan columnas técnicas con demasiados valores nulos, como velocidad de memoria y relojes de funcionamiento.

In [26]:
columnas_modelo = [
    "producto_id",
    "nombre",
    "marca",
    "precio",
    "valoracion_media",
    "total_opiniones",
    "porcentaje_recomendacion",
    "memoria_vram",
    "tipo_memoria",
    "bus_memoria",
]

datos_modelo = datos_gpu[columnas_modelo].copy()

datos_modelo.head()

,producto_id,nombre,marca,precio,valoracion_media,total_opiniones,porcentaje_recomendacion,memoria_vram,tipo_memoria,bus_memoria
0,gpu_0001,Acer Nitro Radeon RX 9060 XT OC 8GB GDDR6 Edic...,Acer,549.00,0.0,0,0.0,8 GB GDDR6,GB GDDR6,128 bits
1,gpu_0002,Acer Nitro Radeon RX 9060 XT OC 8GB GDDR6 HDMI...,Acer,543.00,0.0,0,0.0,NaN,NaN,NaN
2,gpu_0003,Acer Nitro Radeon RX 9070 OC 16GB GDDR6 PCIe 5...,Acer,715.92,0.0,0,0.0,16 GB GDDR6,GB GDDR6,NaN
3,gpu_0004,Acer Predator BiFrost Radeon RX 9070 OC 16GB G...,Acer,802.59,0.0,0,0.0,16 GB GDDR6,GB GDDR6,256 bits
4,gpu_0005,Acer Predator BiFrost Radeon RX 9070 OC 16GB G...,Acer,738.53,0.0,0,0.0,16 GB GDDR6,GB GDDR6,256 bit


## 3. Transformación de variables técnicas

Convertimos la memoria VRAM y el bus de memoria a valores numéricos para poder utilizarlos en el modelo.

Cuando no hay dato disponible, se sustituye por 0 para mantener todos los productos en el análisis.

In [27]:
datos_modelo["memoria_vram_gb"] = (
    datos_modelo["memoria_vram"]
    .str.extract(r"(\d+)")
    .astype(float)
    .fillna(0)
)

datos_modelo["bus_memoria_bits"] = (
    datos_modelo["bus_memoria"]
    .str.extract(r"(\d+)")
    .astype(float)
    .fillna(0)
)

datos_modelo[[
    "memoria_vram",
    "memoria_vram_gb",
    "bus_memoria",
    "bus_memoria_bits",
]].head()

,memoria_vram,memoria_vram_gb,bus_memoria,bus_memoria_bits
0,8 GB GDDR6,8.0,128 bits,128.0
1,NaN,0.0,NaN,0.0
2,16 GB GDDR6,16.0,NaN,0.0
3,16 GB GDDR6,16.0,256 bits,256.0
4,16 GB GDDR6,16.0,256 bit,256.0


## 4. Preparación de variables categóricas

Convertimos el tipo de memoria en columnas numéricas mediante codificación one-hot.

No usamos la marca en el modelo para evitar que los grupos se separen principalmente por fabricante.

In [28]:
datos_categoricos = pd.get_dummies(
    datos_modelo[["tipo_memoria"]],
    dummy_na=True,
    dtype=int,
)

datos_categoricos.head()

,tipo_memoria_GB DDR3,tipo_memoria_GB GDDR2,tipo_memoria_GB GDDR3,tipo_memoria_GB GDDR4,tipo_memoria_GB GDDR5,tipo_memoria_GB GDDR6,tipo_memoria_GB GDDR6 dedicada,tipo_memoria_GB GDDR6X,tipo_memoria_GB GDDR7,tipo_memoria_GB GDDR7 ECC,tipo_memoria_GB GDDR7 a 28 Gbit/s,tipo_memoria_GDDR6,tipo_memoria_GDDR7,tipo_memoria_nan
0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0


## 5. Preparación de la matriz del modelo

Unimos las variables numéricas con las variables categóricas codificadas.

Esta matriz será la entrada del modelo de clustering.

In [29]:
columnas_numericas = [
    "precio",
    "valoracion_media",
    "total_opiniones",
    "porcentaje_recomendacion",
    "memoria_vram_gb",
    "bus_memoria_bits",
]

datos_numericos = datos_modelo[columnas_numericas].copy()

matriz_modelo = pd.concat(
    [
        datos_numericos,
        datos_categoricos,
    ],
    axis=1,
)

print(matriz_modelo.shape)
matriz_modelo.head()

(500, 20)


,precio,valoracion_media,total_opiniones,porcentaje_recomendacion,memoria_vram_gb,bus_memoria_bits,tipo_memoria_GB DDR3,tipo_memoria_GB GDDR2,tipo_memoria_GB GDDR3,tipo_memoria_GB GDDR4,tipo_memoria_GB GDDR5,tipo_memoria_GB GDDR6,tipo_memoria_GB GDDR6 dedicada,tipo_memoria_GB GDDR6X,tipo_memoria_GB GDDR7,tipo_memoria_GB GDDR7 ECC,tipo_memoria_GB GDDR7 a 28 Gbit/s,tipo_memoria_GDDR6,tipo_memoria_GDDR7,tipo_memoria_nan
0,549.00,0.0,0,0.0,8.0,128.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
1,543.00,0.0,0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,715.92,0.0,0,0.0,16.0,0.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,802.59,0.0,0,0.0,16.0,256.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
4,738.53,0.0,0,0.0,16.0,256.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


## 6. Escalado de variables

Escalamos las variables para que todas tengan una importancia comparable dentro del modelo.

Esto evita que variables con valores grandes, como el precio, dominen sobre el resto.

In [30]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()

matriz_escalada = escalador.fit_transform(matriz_modelo)

matriz_escalada.shape

(500, 20)

## 7. Entrenamiento del modelo KMeans

Entrenamos un modelo KMeans para agrupar las tarjetas gráficas en tres grupos.

El número de grupos se fija en 3 para obtener una clasificación sencilla e interpretable.

In [31]:
from sklearn.cluster import KMeans

modelo_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10,
)

datos_modelo["grupo"] = modelo_kmeans.fit_predict(matriz_escalada)

datos_modelo["grupo"].value_counts().sort_index()

grupo
0    137
1    198
2    165
Name: count, dtype: int64

## 8. Análisis de los grupos obtenidos

Analizamos las características medias de cada grupo para entender qué tipo de tarjetas gráficas ha separado el modelo.

In [36]:
resumen_grupos = (
    datos_modelo
    .groupby("grupo")
    .agg(
        productos=("producto_id", "count"),
        precio_medio=("precio", "mean"),
        precio_minimo=("precio", "min"),
        precio_maximo=("precio", "max"),
        vram_media=("memoria_vram_gb", "mean"),
        bus_medio=("bus_memoria_bits", "mean"),
        valoracion_media=("valoracion_media", "mean"),
        opiniones_medias=("total_opiniones", "mean"),
    )
    .round(2)
)

resumen_grupos

,productos,precio_medio,precio_minimo,precio_maximo,vram_media,bus_medio,valoracion_media,opiniones_medias
grupo,,,,,,,,
0,137,1765.46,45.23,19199.99,16.36,179.50,0.40,1.80
1,198,722.33,54.95,2007.56,12.17,154.67,4.63,144.04
2,165,1571.84,40.99,19199.99,0.01,45.38,3.71,94.05


## 9. Revisión de productos por grupo

Mostramos algunos productos de cada grupo para comprobar si la separación obtenida es coherente.

In [33]:
columnas_revision = [
    "producto_id",
    "nombre",
    "marca",
    "precio",
    "memoria_vram",
    "bus_memoria",
    "valoracion_media",
    "total_opiniones",
    "grupo",
]

datos_modelo[columnas_revision].sort_values(
    ["grupo", "precio"],
    ascending=[True, False],
).groupby("grupo").head(5)

,producto_id,nombre,marca,precio,memoria_vram,bus_memoria,valoracion_media,total_opiniones,grupo
382,gpu_0383,Tarjeta Gráfica PNY RTX PRO 6000 Blackwell 96G...,PNY,19199.99,96 GB GDDR7,512 bit,5.0,2,0
381,gpu_0382,Tarjeta Gráfica PNY RTX PRO 6000 Blackwell 96G...,PNY,15606.11,96 GB GDDR7,512 bit,4.5,10,0
383,gpu_0384,PNY RTX PRO 6000 Blackwell Server Edition 96GB...,PNY,12098.79,96 GB GDDR7,512 bits,0.0,0,0
379,gpu_0380,Tarjeta Gráfica PNY RTX PRO 5000 72GB GDDR7 PC...,PNY,11069.40,72 GB GDDR7,384 bits,0.0,0,0
247,gpu_0248,Tarjeta Gráfica Lenovo RTX 5880 Ada 48GB GDDR6...,Lenovo,9609.00,NaN,384 bit,0.0,0,0
48,gpu_0049,ASUS GeForce RTX 5080 NOCTUA OC 16GB GDDR7 Ref...,Asus,2007.56,16 GB GDDR7,256 bit,5.0,57,1
129,gpu_0129,Gigabyte GeForce RTX 5080 AORUS XTREME WATERFO...,Gigabyte,1741.40,16 GB GDDR7,256 bits,4.7,301,1
204,gpu_0204,Gigabyte GeForce RTX 5080 AORUS XTREME WATERFO...,Gigabyte,1741.40,16 GB GDDR7,256 bits,4.7,301,1
338,gpu_0339,Tarjeta Gráfica PNY GeForce RTX 5080 ARGB Over...,PNY,1666.35,16GB GDDR7,256 bits,4.5,36,1
348,gpu_0349,Tarjeta Gráfica PNY GeForce RTX 5080 ARGB Over...,PNY,1666.35,16GB GDDR7,256 bits,4.5,36,1


## 10. Etiquetado de los grupos

Asignamos un nombre descriptivo a cada grupo según las características observadas.

Uno de los grupos recoge productos con información técnica incompleta, por lo que se etiqueta de forma separada.

In [38]:
nombres_grupos = {
    0: "alta_gama",
    1: "gama_media",
    2: "ficha_incompleta",
}

datos_modelo["nombre_grupo"] = datos_modelo["grupo"].map(nombres_grupos)

datos_modelo[[
    "producto_id",
    "nombre",
    "precio",
    "memoria_vram",
    "bus_memoria",
    "grupo",
    "nombre_grupo",
]].head()

,producto_id,nombre,precio,memoria_vram,bus_memoria,grupo,nombre_grupo
0,gpu_0001,Acer Nitro Radeon RX 9060 XT OC 8GB GDDR6 Edic...,549.00,8 GB GDDR6,128 bits,0,alta_gama
1,gpu_0002,Acer Nitro Radeon RX 9060 XT OC 8GB GDDR6 HDMI...,543.00,NaN,NaN,2,ficha_incompleta
2,gpu_0003,Acer Nitro Radeon RX 9070 OC 16GB GDDR6 PCIe 5...,715.92,16 GB GDDR6,NaN,0,alta_gama
3,gpu_0004,Acer Predator BiFrost Radeon RX 9070 OC 16GB G...,802.59,16 GB GDDR6,256 bits,0,alta_gama
4,gpu_0005,Acer Predator BiFrost Radeon RX 9070 OC 16GB G...,738.53,16 GB GDDR6,256 bit,0,alta_gama


## 11. Guardado de los grupos en PostgreSQL

Guardamos el grupo asignado a cada tarjeta gráfica para poder consultarlo después desde la base de datos sin ejecutar nuevamente el modelo.

In [39]:
resultados_cluster_gpu = [
    (
        fila.producto_id,
        int(fila.grupo),
        fila.nombre_grupo,
    )
    for fila in datos_modelo[
        ["producto_id", "grupo", "nombre_grupo"]
    ].itertuples(index=False)
]

assert len(resultados_cluster_gpu) == datos_modelo["producto_id"].nunique()

with conn.cursor() as cursor:
    cursor.executemany(
        """
        INSERT INTO resultados_clustering_gpu (
            producto_id,
            grupo,
            nombre_grupo
        )
        VALUES (%s, %s, %s)
        ON CONFLICT (producto_id) DO UPDATE SET
            grupo = EXCLUDED.grupo,
            nombre_grupo = EXCLUDED.nombre_grupo;
        """,
        resultados_cluster_gpu,
    )

conn.commit()

print(f"Resultados guardados: {len(resultados_cluster_gpu)}")

Resultados guardados: 500


## Conclusión

El modelo ha agrupado las tarjetas gráficas en tres perfiles principales.

El grupo `alta_gama` recoge tarjetas con mayor capacidad técnica y precios más altos.  
El grupo `gama_media` reúne modelos más equilibrados y con mayor presencia de opiniones.  
El grupo `ficha_incompleta` agrupa productos con información técnica insuficiente para compararlos con el resto.

Los resultados se han guardado en PostgreSQL en la tabla `resultados_clustering_gpu`.